## 6. Wetterdaten-Anreicherung via Open-Meteo API

Um den Einfluss von Witterungsbedingungen auf das Rennergebnis modellieren zu können,
wird der Datensatz um historische Wetterdaten erweitert. Für jede Etappe werden sowohl
am **Startort** (`departure`) als auch am **Zielort** (`arrival`) die meteorologischen
Bedingungen zum jeweiligen Renntag abgefragt.

Als Datenquelle dient die kostenfreie [**Open-Meteo Archive API**](https://open-meteo.com/),
die stundengenaue Wetterwerte für beliebige Koordinaten und historische Datumsangaben liefert –
ohne API-Schlüssel.

**Abgefragte Wettervariablen:**

| Variable (API)          | Zielspalte                   | Beschreibung                         |
|-------------------------|------------------------------|--------------------------------------|
| `temperature_2m`        | `*_temp_mittel`              | Temperatur in 2 m Höhe (°C)          |
| `rain`                  | `*_regen_mittel`             | Regen (mm)                           |
| `wind_speed_10m`        | `*_wind_mittel`              | Windgeschwindigkeit in 10 m (km/h)   |
| `relativehumidity_2m`   | `*_luftfeuchte_mittel`       | Relative Luftfeuchtigkeit (%)        |
| `precipitation`         | `*_niederschlag_mittel`      | Gesamtniederschlag (mm)              |
| `winddirection_10m`     | `*_windrichtung_mittel`      | Windrichtung in 10 m (°)             |

**Arbeitsschritte in diesem Notebook:**

1. **Laden** – Einlesen des Datensatzes mit Koordinaten aus dem vorherigen Checkpoint.
2. **Wetterdaten-Funktion** – Eine Hilfsfunktion ruft die stündlichen Wetterdaten für
   eine gegebene Koordinate und ein Datum ab. Bei einem Rate-Limit-Fehler (HTTP 429)
   wird automatisch 60 Minuten gewartet und der Aufruf wiederholt.
3. **Mittelwert-Funktion** – Die stündlichen Rohdaten werden auf einen relevanten
   Zeitfenster-Mittelwert aggregiert: `12–15 Uhr` für den Startort, `15–18 Uhr` für
   den Zielort – als grobe Annäherung an die tatsächliche Rennzeit.
4. **Unique-Kombinations-Optimierung** – Um die Anzahl der API-Aufrufe zu minimieren,
   werden ausschließlich einzigartige `(Koordinate, Datum)`-Kombinationen abgefragt.
   Die Ergebnisse werden anschließend per Left-Join zurück in den Haupt-DataFrame
   gespielt.
5. **Rate-Limiting** – Zwischen jeder Anfrage wird 3 Sekunden gewartet, um das
   API-Limit von max. 20 Anfragen/Minute einzuhalten.
6. **Speichern** – Der vollständig angereicherte Datensatz wird als neues Pickle-File
   persistiert.

> ⚠️ **Hinweis:** Aufgrund des 3-Sekunden-Delays und der Vielzahl einzigartiger
> Etappen-Koordinaten kann dieser Schritt mehrere Stunden in Anspruch nehmen.
> Es empfiehlt sich, den Code unbeaufsichtigt im Hintergrund laufen zu lassen.

In [ ]:
import os
os.chdir(".src/Notebooks")

In [ ]:
import pickle
import requests
import pandas as pd
import time

# 1. Laden
with open("../../data/processed/09_cleaned_master_data.pkl", "rb") as f:
    df = pickle.load(f)

df["date"] = pd.to_datetime(df["date"])

# 2. Wetterdaten-Funktion
def get_wetter(lat, lon, datum):
    if pd.isna(lat) or pd.isna(lon):
        return {}
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude={lat}&longitude={lon}"
        f"&start_date={datum}&end_date={datum}"
        f"&hourly=temperature_2m,rain,wind_speed_10m,"
        f"relativehumidity_2m,precipitation,winddirection_10m"
        f"&timezone=Europe%2FBerlin"
    )
    response = requests.get(url)
    if response.status_code == 200:
        return response.json().get("hourly", {})
    elif response.status_code == 429:
        print("⚠️ Rate Limit – 3600s warten...")
        time.sleep(3600)
        response2 = requests.get(url)
        if response2.status_code == 200:
            return response2.json().get("hourly", {})
    return {}

# 3. Hilfsfunktion: Mittelwert für Stundenbereich 
def mittelwert_stundenbereich(daten, datum, stunden_von, stunden_bis, key):
    zeiten = daten.get("time", [])
    werte  = daten.get(key, [])
    gefiltert = [
        wert for zeit, wert in zip(zeiten, werte)
        if zeit.startswith(datum)
        and stunden_von <= int(zeit[11:13]) <= stunden_bis
        and wert is not None
    ]
    return round(sum(gefiltert) / len(gefiltert), 2) if gefiltert else None

# 4. Stundenbereiche und Variablen 
stunden_config = {
    "departure": (12, 15),
    "arrival":   (15, 18),
}

wetter_variablen = {
    "temperature_2m":      "temp_mittel",
    "rain":                "regen_mittel",
    "wind_speed_10m":      "wind_mittel",
    "relativehumidity_2m": "luftfeuchte_mittel",
    "precipitation":       "niederschlag_mittel",
    "winddirection_10m":   "windrichtung_mittel",
}

# 5. Nur unique Kombinationen abfragen und mergen
for prefix, (std_von, std_bis) in stunden_config.items():
    lat_col = f"{prefix}_lat"
    lon_col = f"{prefix}_lon"

    # Einzigartige Ort+Tag-Kombinationen
    unique = (
        df[[lat_col, lon_col, "date"]]
        .drop_duplicates()
        .dropna()
        .copy()
    )
    unique["datum_str"] = unique["date"].dt.strftime("%Y-%m-%d")

    print(f"{prefix}: {len(unique)} einzigartige API-Anfragen")

    # Wetterdaten nur für unique Kombinationen abrufen
    for api_key, spalten_suffix in wetter_variablen.items():
        unique[f"{prefix}_{spalten_suffix}"] = None

    for i, (idx, row) in enumerate(unique.iterrows()):
        if i % 100 == 0:
            print(f"⏳ {prefix}: {i}/{len(unique)}")

        time.sleep(3)  # 3 Sekunden Pause → max 20 Anfragen/Minute
        daten = get_wetter(row[lat_col], row[lon_col], row["datum_str"])

        for api_key, spalten_suffix in wetter_variablen.items():
            unique.at[idx, f"{prefix}_{spalten_suffix}"] = mittelwert_stundenbereich(
                daten, row["datum_str"], std_von, std_bis, api_key
            )

    # Ergebnisse per Merge zurück in den Haupt-DataFrame
    df = df.merge(
        unique[[lat_col, lon_col, "date"] + [f"{prefix}_{s}" for s in wetter_variablen.values()]],
        on=[lat_col, lon_col, "date"],
        how="left"
    )

    print(f"{prefix} fertig!")

# 6. Speichern
df.to_pickle("../../data/processed/10_cleaned_master_data.pkl")
print(df[[
    "departure", "departure_temp_mittel", "departure_regen_mittel",
    "departure_wind_mittel", "departure_luftfeuchte_mittel",
    "departure_niederschlag_mittel", "departure_windrichtung_mittel",
    "arrival", "arrival_temp_mittel", "arrival_regen_mittel",
    "arrival_wind_mittel", "arrival_luftfeuchte_mittel",
    "arrival_niederschlag_mittel", "arrival_windrichtung_mittel"
]].head())

departure: 1312 einzigartige API-Anfragen
⏳ departure: 0/1312
⏳ departure: 100/1312
⏳ departure: 200/1312
⏳ departure: 300/1312
⏳ departure: 400/1312
⏳ departure: 500/1312
⏳ departure: 600/1312
⏳ departure: 700/1312
⏳ departure: 800/1312
⏳ departure: 900/1312
⏳ departure: 1000/1312
⏳ departure: 1100/1312
⏳ departure: 1200/1312
⏳ departure: 1300/1312
departure fertig!
arrival: 1312 einzigartige API-Anfragen
⏳ arrival: 0/1312
⏳ arrival: 100/1312
⏳ arrival: 200/1312
⏳ arrival: 300/1312
⏳ arrival: 400/1312
⏳ arrival: 500/1312
⏳ arrival: 600/1312
⏳ arrival: 700/1312
⏳ arrival: 800/1312
⏳ arrival: 900/1312
⏳ arrival: 1000/1312
⏳ arrival: 1100/1312
⏳ arrival: 1200/1312
⏳ arrival: 1300/1312
arrival fertig!
    departure departure_temp_mittel departure_regen_mittel  \
0  Fromentine                  20.0                    0.0   
1  Fromentine                  20.0                    0.0   
2  Fromentine                  20.0                    0.0   
3  Fromentine                  20.0         

### Check Data Cleaned_master_data_with_coordinates_and_weather


In [32]:
import pandas as pd
df1 = pd.read_pickle('../../data/processed/10_cleaned_master_data.pkl')


In [ ]:
# Einstellung: Alle Zeilen der Series anzeigen (nicht kürzen)
pd.set_option('display.max_rows', None)

# Missing Values berechnen
missing_stats = df1.isnull().sum()

print("Vollständige Liste der Missing Values pro Spalte:")
print("-" * 50)
print(missing_stats)
print("-" * 50)

# Einstellung zurücksetzen (optional, falls du später wieder kurze Prints willst)
pd.reset_option('display.max_rows')

Vollständige Liste der Missing Values pro Spalte:
--------------------------------------------------
race                                0
year                                0
url                                 0
rank                             2852
rider_url                           0
time_gap                            0
birthdate                        2874
height                           3932
name                             2874
nationality                      2874
weight                           3932
url_name                         2874
departure                           0
arrival                             0
distance                            0
vertical_meters                  5312
profile_score                    5312
won_how                             0
avg_speed                         153
race_ranking                        0
one_day_races                       0
gc                                  0
time_trial                          0
sprint                   